# Week 8 - Day 2: Document Loading and Chunking

## Learning Objectives

By the end of this session, I will be able to:

1. Explain document loading in a RAG system.
2. Understand how text is extracted from PDF files.
3. Load and read a PDF using Python.
4. Understand document metadata.
5. Explain why documents need to be split into chunks.
6. Understand chunk size and chunk overlap.
7. Create text chunks using Python.
8. Prepare document chunks for embeddings and retrieval.

## 1. What Is Document Loading?

Document loading is the process of reading information
from a file or another data source and making it available
for further processing.

In a RAG application, documents may come from:

- PDF files
- Text files
- Word documents
- Websites
- Company knowledge bases

For our Chat with PDF project, the user uploads a PDF.

The application must read the PDF and extract its text
before it can process the document for retrieval.

## 2. Why Is Document Loading Important?

An LLM cannot automatically read a file just because
the file exists on your computer.

A document-processing pipeline must make the file's
contents available to the application.

For a PDF-based RAG system:

1. Receive the PDF.
2. Open and read the PDF.
3. Extract its text.
4. Organize the extracted content.
5. Prepare the content for chunking and embedding.

If the text extraction step fails, later retrieval
steps may not have useful document content.

## 3. Document Loading in the RAG Pipeline
```
PDF
 ↓
Document Loader
 ↓
Extracted Text
 ↓
Text Cleaning
 ↓
Text Chunking
 ↓
Embeddings
 ↓
Vector Database
```
Today, we focus on document loading, text extraction,
and chunking.

Embeddings and vector databases will be studied
in more detail in the next lessons.

In [1]:
document_name = "employee_handbook.pdf"

print("Document to process:", document_name)

Document to process: employee_handbook.pdf


### Observation

The variable stores the name of a PDF.

However, storing a filename does not load or read
the actual document.

We need a PDF-processing library to access its contents.

## 4. What Is Text Extraction?

Text extraction is the process of retrieving readable
text from a document.

Example PDF content:

"Employees receive 12 casual leave days per year."

After text extraction, Python may receive:

Employees receive 12 casual leave days per year.

The extracted text can then be cleaned, divided into
chunks, and processed further.

## 5. Text-Based PDFs vs Scanned PDFs

### Text-Based PDF

A text-based PDF contains selectable text.

A PDF library can often extract its text directly.

### Scanned PDF

A scanned PDF may contain pages stored as images.

A normal text extractor may return little or no text.

Scanned documents may require OCR
(Optical Character Recognition).

OCR attempts to recognize text from images.

Important:
Text extraction quality depends on the document's
format, layout, language, and quality.

## 6. What Is a Document Loader?

A document loader is a component that reads a document
from a source and returns its contents in a usable form.

A loader may provide:

- Extracted text
- Page information
- Source filename
- Other metadata

Different document formats may require different loaders.

For this lesson, we will use PyMuPDF to read PDFs.

In [2]:
%pip install -U pymupdf

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------- ----------------------------- 5.2/19.8 MB 26.9 MB/s eta 0:00:01
   -------------------- ------------------- 10.2/19.8 MB 24.3 MB/s eta 0:00:01
   ------------------------ --------------- 12.1/19.8 MB 20.0 MB/s eta 0:00:01
   -------------------------- ------------- 13.1/19.8 MB 15.4 MB/s eta 0:00:01
   ------------------------------ --------- 14.9/19.8 MB 14.0 MB/s eta 0:00:01
   ----------------------------------- ---- 17.6/19.8 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------  19.7/19.8 MB 13.4 MB/s eta 0:00:01
   ---------------------------------------- 19.8/19.8 MB 12.4 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


## 7. Import the PDF Library

PyMuPDF is a Python library for working with PDF files.

We will use it to:

1. Open a PDF.
2. Access individual pages.
3. Extract text from each page.
4. Keep track of page numbers.

In [3]:
import pymupdf

print("PyMuPDF imported successfully.")

PyMuPDF imported successfully.


## 8. Create a Small Practice Document

To practice document loading, we will create a small
PDF containing sample employee handbook information.

This avoids depending on an external PDF for the
first exercise.

The sample document is only for learning.
It does not represent a real company's policy.

In [4]:
import pymupdf

sample_pdf_path = "sample_employee_handbook.pdf"

sample_pages = [
    """
    Employee Handbook

    Working Hours

    Employees work from Monday to Friday.
    Standard working hours are 9:00 AM to 6:00 PM.
    """,
    """
    Leave Policy

    Employees receive 12 casual leave days per year.
    Leave requests must be approved by a manager.
    """,
    """
    Remote Work Policy

    Employees may work remotely up to two days per week,
    subject to manager approval.
    """
]

doc = pymupdf.open()

for page_text in sample_pages:
    page = doc.new_page()
    page.insert_textbox(
        pymupdf.Rect(50, 50, 550, 750),
        page_text,
        fontsize=12
    )

doc.save(sample_pdf_path)
doc.close()

print("Sample PDF created:", sample_pdf_path)

Sample PDF created: sample_employee_handbook.pdf


### Observation

We created a PDF with three pages.

Each page contains a different topic:

- Working hours
- Leave policy
- Remote work policy

We will now load this PDF and extract its text.

## 9. Open a PDF File

PyMuPDF can open a PDF using pymupdf.open().

The opened document provides access to its pages.

We can then iterate through the pages and extract text.

In [5]:
doc = pymupdf.open(sample_pdf_path)

print("Number of pages:", len(doc))

Number of pages: 3


## 10. Extract Text from One Page

A PDF document contains pages.

We can access a page using its index.

Python uses zero-based indexing:

- First page: index 0
- Second page: index 1
- Third page: index 2

The get_text() method extracts text from a page.

In [6]:
first_page = doc[0]

first_page_text = first_page.get_text()

print(first_page_text)

    Employee Handbook
    Working Hours
    Employees work from Monday to Friday.
    Standard working hours are 9:00 AM to 6:00 PM.



### Observation

The output contains the text extracted from the first
page of the PDF.

The extracted text can now be processed as a Python string.

In [7]:
# Extract text from every page

all_pages_text = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text()

    all_pages_text.append(page_text)

    print(f"Page {page_number}")
    print(page_text)
    print("-" * 40)

Page 1
    Employee Handbook
    Working Hours
    Employees work from Monday to Friday.
    Standard working hours are 9:00 AM to 6:00 PM.

----------------------------------------
Page 2
    Leave Policy
    Employees receive 12 casual leave days per year.
    Leave requests must be approved by a manager.

----------------------------------------
Page 3
    Remote Work Policy
    Employees may work remotely up to two days per week,
    subject to manager approval.

----------------------------------------


## 11. Combine Text from All Pages

We can combine the extracted page text into one
large string.

However, keeping page boundaries is often useful.

For RAG applications, page information can help us
identify where retrieved information came from.

In [8]:
full_text = "\n".join(all_pages_text)

print(full_text)

    Employee Handbook
    Working Hours
    Employees work from Monday to Friday.
    Standard working hours are 9:00 AM to 6:00 PM.

    Leave Policy
    Employees receive 12 casual leave days per year.
    Leave requests must be approved by a manager.

    Remote Work Policy
    Employees may work remotely up to two days per week,
    subject to manager approval.



### Important Observation

We have extracted text from the PDF.

But we have not yet:

- Cleaned the text
- Split it into chunks
- Generated embeddings
- Stored vectors

Document loading is only the first part of the pipeline.

In [9]:
doc.close()

## 12. What Is Document Metadata?

Metadata is additional information associated with
a document or a piece of text.

Examples:

- Source filename
- Page number
- Document title
- Section heading
- Upload date

Metadata is not necessarily the main text content.

It helps organize, identify, filter, and trace documents.

In a Chat with PDF application, page metadata can help
show which PDF page supports an answer.

In [10]:
sample_document = {
    "text": "Employees receive 12 casual leave days per year.",
    "metadata": {
        "source": "sample_employee_handbook.pdf",
        "page": 2,
        "section": "Leave Policy"
    }
}

print(sample_document)

{'text': 'Employees receive 12 casual leave days per year.', 'metadata': {'source': 'sample_employee_handbook.pdf', 'page': 2, 'section': 'Leave Policy'}}


### Observation

The text contains the actual information.

The metadata describes where that information came from.

Keeping both text and metadata is useful when we later
retrieve relevant chunks from a vector database.

## 13. Why Preserve Page Numbers?

Suppose a user asks:

"How many casual leave days are available?"

The system retrieves a chunk from page 2.

If the page number was preserved, the application
can display the source page.

Without metadata, the application may know the text
but have difficulty identifying its original location.

Metadata supports traceability, but it does not
automatically guarantee that a generated answer is correct.

## 14. What Is Document Chunking?

Document chunking is the process of splitting a long
document into smaller pieces of text.

Each smaller piece is called a chunk.

Example:

Original text:
A long employee handbook containing 100 pages.

After chunking:
- Chunk 1
- Chunk 2
- Chunk 3
- ...
- Chunk N

Each chunk contains only part of the original document.

These chunks can later be converted into embeddings
and stored for retrieval.

## 15. Why Do We Need Chunking?

Chunking is useful because:

1. Documents can be very long.
2. A user question usually relates to only part
   of a document.
3. Smaller text pieces can be retrieved individually.
4. Relevant chunks can be supplied to the LLM.
5. Chunking helps manage the amount of context
   sent to the model.

The goal is to create chunks that are small enough
to retrieve efficiently but large enough to preserve
useful meaning.

## 16. Example Without Chunking

Imagine a PDF contains:
```
Page 1: Company introduction
Page 2: Leave policy
Page 3: Remote work policy
Page 4: Salary policy
Page 5: IT security policy
```
Question:
"How many casual leave days are available?"

If the whole document is treated as one large piece,
retrieval may not be as focused.

Chunking allows the system to retrieve the part
that discusses leave policy.

Chunking alone does not guarantee that the correct
section will be retrieved. Retrieval quality matters too.

## 17. Chunking by Characters

One simple strategy is to split text after a fixed
number of characters.

Example:

Chunk size = 100 characters

The text is divided into pieces containing approximately
100 characters each.

This is easy to implement, but it may split a sentence
or paragraph in the middle.

Character-based chunking is useful for understanding
the basic concept.

In [11]:
text = """
RAG stands for Retrieval-Augmented Generation.
It retrieves relevant information from external documents.
The retrieved context is provided to an LLM.
The LLM then generates an answer using that context.
"""

chunk_size = 50

chunks = [
    text[i:i + chunk_size]
    for i in range(0, len(text), chunk_size)
]

for index, chunk in enumerate(chunks, start=1):
    print(f"Chunk {index}:")
    print(repr(chunk))
    print()

Chunk 1:
'\nRAG stands for Retrieval-Augmented Generation.\nIt'

Chunk 2:
' retrieves relevant information from external docu'

Chunk 3:
'ments.\nThe retrieved context is provided to an LLM'

Chunk 4:
'.\nThe LLM then generates an answer using that cont'

Chunk 5:
'ext.\n'



### Observation

The text was split into chunks of up to 50 characters.

Notice that some chunks may begin or end in the middle
of a word or sentence.

This can make the text harder to understand.

More advanced chunking strategies attempt to preserve
sentence or paragraph boundaries.

## 18. What Is Chunk Size?

Chunk size is the amount of text placed into each chunk.

It may be measured in:

- Characters
- Words
- Tokens

The appropriate size depends on the document, embedding
model, retrieval strategy, and LLM context limits.

### Small Chunks

Advantages:
- More focused retrieval
- Less unrelated text per chunk

Disadvantages:
- May lose surrounding context
- Important information may be split across chunks

### Large Chunks

Advantages:
- Preserve more surrounding context
- May keep related information together

Disadvantages:
- May include irrelevant information
- Can use more context when sent to an LLM

## 19. What Is Chunk Overlap?

Chunk overlap is the amount of text shared between
consecutive chunks.

Example:

Chunk size = 100 characters

Chunk overlap = 20 characters

The next chunk begins 20 characters before the
previous chunk ends.

Overlap helps preserve context near chunk boundaries.

However, too much overlap can create unnecessary
duplication and increase storage and processing.

## 20. Chunking Example

Original text:

ABCDEFGHIJKLMNO
```
Chunk size = 6
Overlap = 2
```
Possible chunks:
```
Chunk 1: ABCDEF
Chunk 2: EFGHIJ
Chunk 3: IJKLMN
Chunk 4: MNO
```
The shared characters help illustrate overlap.

In real documents, we usually want to preserve
meaningful text boundaries rather than splitting
arbitrary letters.

In [12]:
def create_chunks(text, chunk_size, overlap):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


example_text = "ABCDEFGHIJKLMNO"

example_chunks = create_chunks(
    example_text,
    chunk_size=6,
    overlap=2
)

for i, chunk in enumerate(example_chunks, start=1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: ABCDEF
Chunk 2: EFGHIJ
Chunk 3: IJKLMN
Chunk 4: MNO


### Observation

The chunks share some characters.

This demonstrates how overlap works.

Our function uses character-based splitting for
learning purposes. It does not yet account for
sentences, paragraphs, or token limits.

## 21. Why Preserve Sentence Boundaries?

A sentence often contains a complete thought.

If a chunk splits a sentence in the middle, the
retrieved text may be harder to interpret.

For example:

Bad split:
"Employees may work remotely up to two"

"days per week, subject to manager approval."

Better:
"Employees may work remotely up to two days per week,
subject to manager approval."

Sentence-aware chunking tries to keep related text
together.

However, a single very long sentence may still need
to be split.

In [13]:
import re

sample_text = """
RAG retrieves relevant information.
Embeddings represent text as vectors.
Chunking divides documents into smaller pieces.
"""

sentences = re.split(r"(?<=[.!?])\s+", sample_text.strip())

for sentence in sentences:
    print(sentence)

RAG retrieves relevant information.
Embeddings represent text as vectors.
Chunking divides documents into smaller pieces.


## 22. Create Chunks by Grouping Sentences

A simple strategy is to group sentences until
the combined character count reaches a target size.

This helps preserve sentence boundaries.

It is still a simplified approach because it does
not understand the full semantic meaning of a document.

In [14]:
def sentence_chunks(text, max_chars=100):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if not sentence:
            continue

        if current_chunk and len(current_chunk) + len(sentence) + 1 > max_chars:
            chunks.append(current_chunk)
            current_chunk = sentence
        else:
            if current_chunk:
                current_chunk += " "
            current_chunk += sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


sentence_chunk_list = sentence_chunks(
    sample_text,
    max_chars=80
)

for i, chunk in enumerate(sentence_chunk_list, start=1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: RAG retrieves relevant information. Embeddings represent text as vectors.
Chunk 2: Chunking divides documents into smaller pieces.


### Observation

This function groups complete sentences when possible.

It does not split an individual sentence that exceeds
the maximum size.

A production chunking pipeline needs additional handling
for long sentences, headings, tables, and other layouts.

## 23. Combine Document Loading and Chunking

Now we will combine the concepts from today's lesson.

Pipeline:
```
PDF
 ↓
Load PDF
 ↓
Extract Text Page by Page
 ↓
Create Chunks
 ↓
Attach Metadata
 ↓
Prepare for Embeddings
```
We will preserve the source filename and page number
for each chunk.

In [15]:
import pymupdf
import re

def load_pdf_pages(pdf_path):
    documents = []

    with pymupdf.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text().strip()

            if text:
                documents.append({
                    "text": text,
                    "metadata": {
                        "source": pdf_path,
                        "page": page_number
                    }
                })

    return documents


loaded_pages = load_pdf_pages(sample_pdf_path)

print("Pages with extracted text:", len(loaded_pages))

Pages with extracted text: 3


In [17]:
def split_into_sentence_chunks(text, max_chars=250):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if not sentence:
            continue

        if len(sentence) > max_chars:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            # Simple fallback for an unusually long sentence
            for start in range(0, len(sentence), max_chars):
                chunks.append(sentence[start:start + max_chars])

        elif current_chunk and len(current_chunk) + len(sentence) + 1 > max_chars:
            chunks.append(current_chunk)
            current_chunk = sentence

        else:
            current_chunk = (
                current_chunk + " " + sentence
                if current_chunk
                else sentence
            )

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [18]:
def chunk_documents(documents, max_chars=250):
    chunked_documents = []

    for document in documents:
        text_chunks = split_into_sentence_chunks(
            document["text"],
            max_chars=max_chars
        )

        for chunk_index, chunk_text in enumerate(text_chunks, start=1):
            chunked_documents.append({
                "text": chunk_text,
                "metadata": {
                    **document["metadata"],
                    "chunk": chunk_index
                }
            })

    return chunked_documents


chunks = chunk_documents(
    loaded_pages,
    max_chars=150
)

print("Total chunks:", len(chunks))

Total chunks: 3


In [19]:
# Display the chunks with their metadata

for chunk in chunks:
    print("Text:", chunk["text"])
    print("Metadata:", chunk["metadata"])
    print("-" * 50)

Text: Employee Handbook
    Working Hours
    Employees work from Monday to Friday. Standard working hours are 9:00 AM to 6:00 PM.
Metadata: {'source': 'sample_employee_handbook.pdf', 'page': 1, 'chunk': 1}
--------------------------------------------------
Text: Leave Policy
    Employees receive 12 casual leave days per year. Leave requests must be approved by a manager.
Metadata: {'source': 'sample_employee_handbook.pdf', 'page': 2, 'chunk': 1}
--------------------------------------------------
Text: Remote Work Policy
    Employees may work remotely up to two days per week,
    subject to manager approval.
Metadata: {'source': 'sample_employee_handbook.pdf', 'page': 3, 'chunk': 1}
--------------------------------------------------


### Observation

We now have a list of document chunks.

Each chunk contains:

- Text
- Source filename
- Page number
- Chunk number

This structure is useful because the next stages
of a RAG system need both the text and its source
information.

These chunks are not yet stored in a vector database.

## 24. Prepare Text for the Embedding Model

An embedding model takes text as input.

We can extract the text from our chunk records
and pass it to an embedding model.

For example:

[
    "Employees work from Monday to Friday.",
    "Employees receive 12 casual leave days per year."
]

Each text chunk can be converted into a vector.

The metadata should be preserved separately so it
can be associated with the corresponding vector.

In [20]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_metadata = [chunk["metadata"] for chunk in chunks]

print("Number of texts:", len(chunk_texts))
print("Number of metadata records:", len(chunk_metadata))

Number of texts: 3
Number of metadata records: 3


### Important Note

We are preparing the chunks for embeddings, but
we are not generating embeddings in this cell.

The next stage of the project will convert the
chunk texts into vectors and store them for retrieval.

## 25. Practice Exercises

### Exercise 1: Document Loading
Load the sample PDF and print the text from page 2.

### Exercise 2: Metadata
Create a dictionary containing:
- Text
- Source filename
- Page number

### Exercise 3: Character Chunking
Split a paragraph into chunks of 40 characters.

### Exercise 4: Overlap
Use a chunk size of 50 and overlap of 10.
Print the resulting chunks.

### Exercise 5: Sentence Chunking
Create chunks from a paragraph while trying to
preserve sentence boundaries.

### Exercise 6: PDF Pipeline
Load the sample PDF, create chunks, and print
each chunk with its page metadata.

### Exercise 7: Reflection
Explain why chunking the entire PDF into one
large text block may be less useful for retrieval.

## 26. Interview Questions and Answers

### Q1. What is document loading?

Document loading is the process of reading a file
or data source and making its contents available
for further processing.

### Q2. What is text extraction?

Text extraction retrieves readable text from a
document such as a PDF.

### Q3. What is OCR?

OCR stands for Optical Character Recognition.
It attempts to recognize text in images, including
scanned document pages.

### Q4. What is document chunking?

Chunking splits a large document into smaller pieces
that can be processed and retrieved individually.

### Q5. Why is chunking used in RAG?

It helps retrieve focused pieces of information
instead of requiring the system to use an entire
large document for every question.

### Q6. What is chunk size?

Chunk size is the amount of text contained in a chunk.
It may be measured in characters, words, or tokens.

### Q7. What is chunk overlap?

Chunk overlap is the text shared between consecutive
chunks to help preserve context across boundaries.

### Q8. Why is metadata important?

Metadata records information such as source filename
and page number. It helps identify and trace retrieved
information back to its original document.

### Q9. What is a limitation of character-based chunking?

It can split sentences or important phrases in the
middle, potentially making retrieved text harder
to interpret.

### Q10. Does chunking guarantee accurate RAG answers?

No. Chunking is one part of the pipeline. Retrieval,
document quality, context construction, and generation
also affect answer quality.

## 27. Day 2 Cheat Sheet

| Concept | Meaning |
|---|---|
| Document loading | Reading a document into an application |
| Text extraction | Getting readable text from a document |
| OCR | Recognizing text in images |
| Metadata | Information describing a document or chunk |
| Chunking | Splitting a document into smaller pieces |
| Chunk size | Amount of text in a chunk |
| Chunk overlap | Shared text between adjacent chunks |
| Sentence-aware chunking | Attempts to preserve sentence boundaries |

### Today's Pipeline
```
PDF
 ↓
Load Document
 ↓
Extract Text
 ↓
Preserve Metadata
 ↓
Clean Text
 ↓
Create Chunks
 ↓
Prepare Chunks for Embeddings